In [20]:
import pandas as pd
from datetime import datetime
import os
from glob import glob

In [21]:
path="/mnt/c/Users/isabe/OneDrive/Desktop/Lavoro/Tuscia/chiara/peaks_background/bufale_isa/"
os.chdir(path)

In [24]:
months = glob("./dati_bufale/data_to_merge/*.csv", recursive=True)

In [25]:
months

['./dati_bufale/data_to_merge/1_Combined_MooLogger_Data_maggio.csv',
 './dati_bufale/data_to_merge/2_giugno_bufale-05302024000000-06142024235900.csv',
 './dati_bufale/data_to_merge/3_Amaseno Luglio 2024-07012024115100-07312024115100.csv',
 './dati_bufale/data_to_merge/4_Combined_MooLogger_agosto.csv',
 './dati_bufale/data_to_merge/5_settembre_bufale-09072024000000-09102024235900.csv']

In [26]:
maggio = pd.read_csv(months[0])
giugno = pd.read_csv(months[1], skiprows=1)
luglio = pd.read_csv(months[2], skiprows=1)
agosto = pd.read_csv(months[3])
settembre = pd.read_csv(months[4], skiprows=1)


/tmp/ipykernel_21107/3338158241.py:2: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  giugno = pd.read_csv(months[1], skiprows=1)
/tmp/ipykernel_21107/3338158241.py:3: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  luglio = pd.read_csv(months[2], skiprows=1)
/tmp/ipykernel_21107/3338158241.py:5: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  settembre = pd.read_csv(months[4], skiprows=1)


In [37]:
settembre.head()

,timestamp,CH4,CH4-mgm3,CO2,CO2-mgm3,FLOW,H-AIR,P,T-AIR,T-BOX
0,09/07/2024 08:21:02.082181,127.6,83.71,528.6,951.485,0.545,72.16,1006.962,23.778,23.118
1,09/07/2024 08:21:03.084181,127.5,83.645,526.3,947.345,0.847,72.203,1007.039,23.779,23.118
2,09/07/2024 08:21:04.087254,128.1,84.038,521.5,938.705,0.895,72.252,1007.084,23.78,23.118
3,09/07/2024 08:21:05.088786,127.0,83.317,521.4,938.525,1.056,72.313,1007.104,23.781,23.118
4,09/07/2024 08:21:06.091306,127.3,83.513,528.0,950.405,0.789,72.355,1007.099,23.782,23.118


In [12]:
def load_and_clean_csv_moologger(file_path):
    try:
        # Carica il file CSV, impostando tutti i dati come stringhe per evitare problemi di conversione automatica.
        df = pd.read_csv(file_path, skiprows=1)
        print(df.loc[138695:138700])

        # Converte la colonna 'timestamp' da stringa a datetime per facilitare operazioni basate sul tempo.
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        # Definisce un elenco di colonne che dovrebbero contenere valori numerici.
        numeric_columns = ['CH4', 'CH4-mgm3', 'CO2', 'CO2-mgm3', 'FLOW', 'H-AIR', 'P', 'T-AIR', 'T-BOX']

        # Per ogni colonna nell'elenco, tenta di convertire i valori in float. 
        # Qualsiasi valore che non può essere convertito in un numero (ad es., una stringa) viene sostituito con NaN.
        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Rimuove le righe duplicate basandosi sulla colonna 'timestamp' per evitare ridondanze nei dati.
        df = df.drop_duplicates(subset=['timestamp'])

        return df

    except pd.errors.EmptyDataError:
        # Gestisce il caso in cui il file CSV è vuoto o ha una formattazione inaspettata, stampando un messaggio di errore.
        print(f"Il file {file_path} è vuoto o ha una formattazione inaspettata.")
        return pd.DataFrame()
    except Exception as e:
        # Cattura e segnala qualsiasi altro errore incontrato durante il caricamento o la pulizia dei dati.
        print(f"Errore nel caricamento del file {file_path}: {e}")
        return pd.DataFrame()

In [13]:
# Definisce il percorso della cartella contenente i file CSV da processare.
folder_path_moologger = './dati_bufale/data_to_merge/'

# Inizializza un DataFrame vuoto per combinare i dati di tutti i file CSV.
combined_df_moologger = pd.DataFrame()

# Scorre tutti i file nella cartella specificata, processando solo quelli che terminano con '.csv'.
for file_name in os.listdir(folder_path_moologger):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path_moologger, file_name)
        print(file_path)
        try:
            # Carica e pulisce i dati di ciascun file CSV, poi li combina in un unico DataFrame.
            df_cleaned = load_and_clean_csv_moologger(file_path)
            combined_df_moologger = pd.concat([combined_df_moologger, df_cleaned], ignore_index=True)
        except Exception as e:
            # Segnala eventuali errori incontrati durante il processo di un file specifico.
            print(f"Error processing {file_name}: {e}")

# Rimuove ulteriori righe duplicate nel DataFrame combinato, basandosi ancora sulla colonna 'timestamp'.
combined_df_moologger = combined_df_moologger.drop_duplicates(subset=['timestamp'])

# Salva il DataFrame combinato e pulito in un nuovo file CSV.
output_file_moologger = os.path.join('./dati_bufale/', 'Combined_MooLogger_estate.csv')
combined_df_moologger.to_csv(output_file_moologger, index=False)

print("File CSV combinato creato con successo.")



./dati_bufale/data_to_merge/1_Combined_MooLogger_Data_maggio.csv
        2024-05-15 22:00:00.066998  1177.8  772.679  2121.7  3819.079  0.659  \
138695  2024-05-16 17:18:49.037850   896.2  587.939   821.3  1476.187  1.130   
138696  2024-05-16 17:18:49.538853   898.5  589.973   800.0  1440.007  0.721   
138697  2024-05-16 17:18:50.040196   904.0  593.057   777.1  1398.787  0.675   
138698  2024-05-16 17:18:50.541473   904.8  593.581   759.5  1365.307  1.196   
138699  2024-05-16 17:18:51.042917   897.2  588.595   753.6  1356.487  1.021   
138700  2024-05-16 17:18:51.544654   899.1  589.842   748.2  1346.767  0.814   

         40.67  995.509  32.468  32.252  
138695  36.672  993.681  35.541  35.385  
138696  36.660  993.710  35.540  35.385  
138697  36.654  993.723  35.540  35.385  
138698  36.642  993.689  35.540  35.385  
138699  36.636  993.658  35.540  35.385  
138700  36.623  993.674  35.540  35.385  
Errore nel caricamento del file ./dati_bufale/data_to_merge/1_Combined_MooLogger

/tmp/ipykernel_21107/2858605126.py:4: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=1)


                         timestamp    CH4 CH4-mgm3     CO2  CO2-mgm3   FLOW  \
138695  05/31/2024 12:37:19.058643  811.1  532.111  1357.9  2444.232  1.041   
138696  05/31/2024 12:37:20.061712  845.9  554.941  1470.7  2647.273  0.797   
138697  05/31/2024 12:37:21.063587  829.3  544.051  1410.8  2539.453  0.863   
138698  05/31/2024 12:37:22.066165  861.3  565.044  1294.5  2330.112  0.713   
138699  05/31/2024 12:37:23.068946  822.3  539.458  1251.9  2253.431  0.674   
138700  05/31/2024 12:37:24.072268  827.6  542.935  1173.2  2111.771  0.618   

         H-AIR        P   T-AIR   T-BOX  
138695  36.922  995.945  32.931  32.806  
138696  36.922  995.935  32.932  32.806  
138697  36.916  995.917  32.932  32.806  
138698  36.916  995.934  32.932  32.806  
138699  36.916  995.961  32.933  32.806  
138700  36.916  995.911  32.934  32.806  
./dati_bufale/data_to_merge/3_Amaseno Luglio 2024-07012024115100-07312024115100.csv


/tmp/ipykernel_21107/2858605126.py:4: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=1)


                         timestamp   CH4 CH4-mgm3    CO2  CO2-mgm3   FLOW  \
138695  07/03/2024 00:28:18.804252  46.1   29.915  804.7  1448.467  0.606   
138696  07/03/2024 00:28:19.806548  48.0    31.49  800.1  1440.187  0.696   
138697  07/03/2024 00:28:20.809853  46.4    30.44  799.5  1439.107  0.843   
138698  07/03/2024 00:28:21.812920  45.0   29.522  796.4  1433.527  0.854   
138699  07/03/2024 00:28:22.816179  42.5   27.882  789.1  1420.387  0.599   
138700  07/03/2024 00:28:23.819344  63.2   41.461  804.8  1448.647  0.776   

         H-AIR        P   T-AIR   T-BOX  
138695  38.424   994.01  34.165  34.035  
138696  38.424  994.008  34.166  34.035  
138697  38.424  994.032  34.165  34.035  
138698   38.43  994.083  34.165  34.035  
138699  38.436  994.099  34.165  34.035  
138700  38.442  994.073  34.164  34.035  
./dati_bufale/data_to_merge/4_Combined_MooLogger_agosto.csv
        2024-08-01 09:25:28.260847  -123.9  -81.283  600.5  1080.905  0.865  \
138695  2024-08-06 06:24:30

/tmp/ipykernel_21107/2858605126.py:4: DtypeWarning: Columns (1,2,3,4,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=1)


                         timestamp    CH4 CH4-mgm3     CO2  CO2-mgm3   FLOW  \
138695  09/08/2024 23:29:46.348205  294.4  193.137  3612.6  6502.713  0.722   
138696  09/08/2024 23:29:47.351052  280.0   183.69  3364.4   6055.95  1.212   
138697  09/08/2024 23:29:48.352612  266.8   175.03  3237.0  5826.629  0.859   
138698  09/08/2024 23:29:49.354432  257.1  168.667  3126.8  5628.268  1.049   
138699  09/08/2024 23:29:50.357368  238.5   155.94  2762.9  4973.245  0.884   
138700  09/08/2024 23:29:51.360594  209.0  137.112  2263.5   4053.44  1.166   

         H-AIR        P   T-AIR   T-BOX  
138695  45.627  992.567  37.115  36.946  
138696  45.633  992.609  37.114  36.946  
138697  45.639  992.621  37.115  36.946  
138698  45.639  992.589  37.115  36.946  
138699  45.645  992.535  37.115  36.946  
138700  45.645  992.515  37.114  36.946  
File CSV combinato creato con successo.


In [14]:
df = pd.read_csv('./dati_bufale/Combined_MooLogger_estate.csv')

In [15]:
df.head()

,timestamp,CH4,CH4-mgm3,CO2,CO2-mgm3,FLOW,H-AIR,P,T-AIR,T-BOX
0,2024-05-29 22:00:00.584605,848.9,556.909,1371.3,2468.352,0.519,45.181,999.646,30.240,30.057
1,2024-05-29 22:00:01.586117,829.9,544.444,1238.0,2228.411,0.647,45.181,999.684,30.241,30.057
2,2024-05-29 22:00:02.588899,820.3,538.146,1155.9,2080.630,1.061,45.181,999.704,30.240,30.057
3,2024-05-29 22:00:03.590745,811.1,532.111,1121.1,2017.990,0.539,45.181,999.685,30.241,30.057
4,2024-05-29 22:00:04.592643,796.2,522.336,1073.3,1931.950,0.949,45.181,999.640,30.240,30.057
